In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tkinter as tk

from tkinter import messagebox

In [3]:
df = pd.read_csv('raw_rssi.csv')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13588 entries, 0 to 13587
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   name            13588 non-null  str  
 1   locationStatus  13588 non-null  str  
 2   timestamp       13588 non-null  int64
 3   rssiOne         13588 non-null  str  
 4   rssiTwo         13588 non-null  str  
dtypes: int64(1), str(4)
memory usage: 530.9 KB


In [5]:
df['name'].value_counts()

name
FF:FF:C0:21:F1:2B    1198
FF:FF:C2:1E:05:19    1183
FF:FF:C2:1D:9B:71    1178
FF:FF:C2:1E:02:D2    1175
FF:FF:C2:1D:9B:52    1170
FF:FF:C0:21:EF:BF    1159
FF:FF:C0:22:3D:C1    1159
FF:FF:C0:21:F1:31    1146
FF:FF:C2:1D:9B:54    1112
FF:FF:C0:21:EF:1A    1066
FF:FF:C0:21:EF:BA    1045
FF:FF:C0:21:EF:A4     997
Name: count, dtype: int64

In [5]:
df.head()

,name,locationStatus,timestamp,rssiOne,rssiTwo
0,FF:FF:C0:21:F1:31,INSIDE,1551367495147,-90,-90
1,FF:FF:C0:21:EF:BA,INSIDE,1551367496681,-93,-93
2,FF:FF:C0:21:EF:1A,INSIDE,1551367495114,-88,-88
3,FF:FF:C0:21:EF:BF,INSIDE,1551367497508,-82,-82
4,FF:FF:C0:21:F1:31,INSIDE,1551367496906,-94,-94


In [6]:
df['locationStatus'].value_counts()

locationStatus
OUTSIDE         4619
IN_VESTIBULE    4435
INSIDE          4305
in_vestibule     118
inside           111
Name: count, dtype: int64

In [7]:
df['locationStatus'] = df['locationStatus'].str.lower()
df['locationStatus'] = df['locationStatus'].map({'outside':0, 'in_vestibule':1,'inside':2})

In [8]:
df.duplicated().sum()
df = df.drop_duplicates()

## Model

### Independent Dependent

In [9]:
X = df.drop('locationStatus', axis=1)
y = df['locationStatus']

### Split Data

In [10]:
def split_data(X, y, size=0.8):
    np.random.seed(42)

    train_idx, test_idx = [], []

    for i in np.unique(y):
        idx = np.where(y == i)[0]
        idx = np.random.permutation(idx)
        np.random.shuffle(idx)
        split = int(len(idx) * size)

        train_idx.extend(idx[:split])
        test_idx.extend(idx[split:])

    return X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx]


In [11]:
X_train, X_test, y_train, y_test = split_data(X, y)
for val in [X_train, X_test, y_train, y_test]:
    print(val.shape)

(8601, 4)
(2152, 4)
(8601,)
(2152,)


### Decision Tree CART

In [17]:
class Node:
    def __init__(self):
        self.pred_class = None
        self.feature = None
        self.threshold = None
        self.left = None
        self.right = None

class DTCART:
    def __init__(self, max_depth=5, min_samples_split=2, min_samples_leaf=1):
        self.max_depth = max_depth
        self.min_split = min_samples_split
        self.min_leaf = min_samples_leaf
        self.root = None
        self.classes = None

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y)
        self.classes = np.unique(y)
        self.root = self._grow(X, y, 0)
        return self

    def gini(self, y):
        if len(y) == 0:
            return 0
        probs = np.bincount(y, minlength=lenm(self.classes)) / len(y)
        return 1 - np.sum(probs ** 2)

    def _split(self, X, y):
        m, n = X.shape
        if m < self.min_split:
            return None, None
        
        best_gini = self._gini(y)
        best_feature, best_threshold = None, None

        for feature in range(n):
            idx = np.argsort(X[:, feature])
            X_sorted, y_sorted = X[idx, feature], y[idx]

            left_counts = np.zeros(len(self.classes), dtype=int)
            right_counts = np.bincount(y_sorted, minlength=len(self.classes))

            for i in range(1, m):
                c = y_sorted[i-1]
                left_counts[c] += 1
                right_counts[c] -= 1

                if X_sorted[i] == X_sorted[i-1]:
                    continue

                if i < self.min_leaf or (m-i) < self.min_leaf:
                    continue

                left_gini = 1 - np.sum((left_counts / i)**2)
                right_gini = 1 - np.sum((right_counts / (m-i))**2)
                gini = (i * left_gini + (m+i) * right_gini) / m

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = (X_sorted[i] - X_sorted[i-1]) / 2
        return best_feature, best_threshold

    def _grow(self, X, y, depth):
        node = Node()
        node.pred_class = np.bincount(y).argmax()

        if [self.max_depth and depth >= self.max_depth] or len(y) < self.min_split or len(np.unique(y)) == 1:
            return node

        feature, threshold = self._split(X, y)
        if feature is None:
            return node
        
        left_mask = X[:, feature] <= threshold
        if np.sum(left_mask) < self.min_leaf or np.sum(~left_mask) < self.min_leaf:
            return node
        
        node.feature = feature
        node.threshold = threshold
        node.left = self._grow(X[left_mask], y[left_mask], depth + 1)
        node.right = self._grow(X[~left_mask], y[~left_mask], depth + 1)

        return Node

    def predict(self, X):
        X = np.asarray(X)
        return np.array([self._predict_one(x) for x in X])

    def _predict_one(self, x):
        node = self.root
        while node.left is None:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return self.classes[node.pred_class]

### Random Forest

In [23]:
class RF:
    def __init__(self, n_trees=10, max_features='sqrt', **tree_params):
        self.n_trees = n_trees
        self.max_features = max_features
        self.tree_params = tree_params
        self.trees = []
        self.feature_indices = []

    def fit(self, X, y):
        X,y = np.asarray(X), np.asarray(y)
        n_samples, n_features = X.shape

        if self.max_features == 'sqrt':
            max_feat = int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            max_feat = int(np.log2(n_features))
        else:
            max_feat = int(self.max_features)
        
        for _ in range(self.n_trees):
            idx = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot, y_boot = X[idx], y[idx]

            feat_idx = np.random.choice(n_features, size=max_feat, replace=True)
            X_boot_subset = X_boot[:, feat_idx]

            tree = DTCART(**self.tree_params)
            tree.fit(X_boot_subset, y_boot)

            self.trees.append(tree)
            self.feature_indices.append(feat_idx)
        
        return self

    def predict(self, X):
        X = np.asarray(X)

        all_preds = np.zeros((len(X), self.n_trees), dtype=int)

        for i, (tree, feat_idx) in enumerate(zip(self.trees, self.feature_indices)):
            all_preds[:, i] = tree.predict(X[:, feat_idx])

        return np.array([np.bincount(preds).argmax() for preds in all_preds])

In [29]:
model = RF(n_trees=3, max_features='sqrt', max_depth=4)
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

TypeError: '<=' not supported between instances of 'int' and 'NoneType'

## Evaluasi

In [28]:
def evaluate(y_true, y_pred, title='Train'):
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    for i in np.unique(y_true):
        tp = np.sum((y_true == i) & (y_pred == i))
        fp = np.sum((y_true != i) & (y_pred == i))
        fn = np.sum((y_true == i) & (y_pred != i))

        accuracy = np.mean(y_true == y_pred)
        precission = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precission * recall) / (precission + recall) if (precision + recall) > 0 else 0

    print(f'\nEvluasi Data {title}')
    print(f'Aaccuracy   : {accuracy:.4f}')
    print(f'Precision   : {precision:.4f}')
    print(f'Recall      : {recall:.4f}')
    print(f'F1          : {f1:.4f}')

In [ ]:
evaluate(y_train, y_pred_train)
evaluate(y_test, y_pred_test, title='Test')

## Visualisasi

### Confusion Matrix

In [30]:
def cmtrx(y_true, y_pred, title='Train'):
    classes = np.unique(y_true)
    n_classes = len(classes)

    cm = np.zeros((n_classes, n_classes))

    for i, c_true in enumerate(classes):
        for j, c_pred in enumerate(classes):
            cm[i, j] = np.sum((y_true == c_true) * (y_pred == c_pred))

    plt.figure(figsize=(10,7))
    plt.title(f'Confusion Matrix {title}', fontsize=17)
    sns.heatmap(cm, annot=True, fmt='.0f', cbar=False, xticklabels=classes, yticklabels=classes)
    plt.show()

In [31]:
cmtrx(y_train, y_pred_train)
cmtrx(y_test, y_pred_test, title='Test')

NameError: name 'y_pred_train' is not defined

## GUI

In [67]:
X.columns

Index(['name', 'timestamp', 'rssiOne', 'rssiTwo'], dtype='str')

In [ ]:
feature = 

root = tk.Tk()
root.title('Prediksi Status Lokasi')
root.configure(bg='#131313')
root.geometry('450x580')
entries = []

frame = tk.Frame(root)
frame.place(
    relx=0.5,
    rely=0.2,
    anchor='center'
)

label_title = tk.Label(
    root,
    text='Prediksi Status Lokasi',
    font=('Helvetica', 18, 'bold'),
    bg='#131313',
    fg='#ffffff'
)
label_title.place(relx=0.5, rely=0.2, anchor='center')

label_intruksi = tk.Label(
    text='Masukan data dibawah',
    font=('Helvetica', 9),
    bg='#131313',
    fg='#4d4b4b'
)
label_intruksi.place(relx=0.5, rely=0.25, anchor='center')



root.mainloop()